In [1]:
# Google Colab / local setup
import os, sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODE_ROOT = Path("/content/drive/MyDrive/msc project/code")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
        str(CODE_ROOT), 'catboost==1.2.10', 'rapidfuzz==3.14.3',
    ])
else:
    CODE_ROOT = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    )
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
CODE = CODE_ROOT

# 06d - LLM sentiment

## Methodology

A frozen LLM scorer processes first-seen text with batch size 10 before each M15 decision. LLM matched uses the same five causal fields, while LLM full adds three structured fields, with H1 2025 calibration and July 2025-March 2026 descriptive economics.

USA500 and USATECH are separate. Every economic table reports Net, Sharpe,
Sortino and drawdown after a fixed all-in round-trip deduction: 2 bps (0.02%)
for USA500 and 3 bps (0.03%) for USATECH. This covers bid-ask spread plus
commission/slippage; bar-specific spreads remain diagnostic, not additional
costs. Inputs stop before `2026-04-01 00:00 UTC`; Q2-2026 is evaluated separately in Notebook 07 and
no model is refitted here.

In [2]:
import json
import numpy as np
import pandas as pd
from html import escape
from IPython.display import HTML, display
from experiments.build_notebook_06 import MODEL_LABELS, validate_llm_score_manifest
from experiments.index_replication_protocol import MODEL_NAMES
from sentiment.index_scoring import _canonical_hash
from sentiment.score_llm import SYSTEM, _ITEM

STREAMS = ("usa500", "usatech")
ROOT = CODE_ROOT / "experiments" / "cache" / "index_replication"
FORWARD_ROOT = CODE_ROOT / "experiments" / "cache" / "index_all_model_forward"
RAW = CODE_ROOT / "sentiment" / "raw"
identity_files = list(RAW.glob("index_*_identity.json"))
assert len(identity_files) == 1
identity = json.loads(identity_files[0].read_text(encoding="utf-8"))["identity"]
assert identity["tag"] and identity["digest"]
assert identity["temperature"] == 0.0 and identity["think"] == "low"
assert identity["batch_size"] == 10
selected, forward = {}, {}
for stream in STREAMS:
    for prefix in ("", "direct_events_"):
        manifest = json.loads((RAW / f"scores_llm_{prefix}{stream}.manifest.json").read_text(encoding="utf-8"))
        validate_llm_score_manifest(manifest, identity)
    selected[stream] = pd.read_parquet(ROOT / stream / "h1_selected_policies.parquet")
    diagnostic = json.loads((FORWARD_ROOT / stream / "result.json").read_text(encoding="utf-8"))
    assert diagnostic["q2_loaded"] is False and diagnostic["forward_rows"] == 36
    assert diagnostic["evidence_role"] == "secondary_reused_forward_diagnostic"
    forward[stream] = pd.read_parquet(FORWARD_ROOT / stream / "forward_summary.parquet")

registered_arms = set(selected[STREAMS[0]]["arm"])
LLM_FULL = next(arm for arm in registered_arms if arm.endswith("_full"))
llm_prefix = LLM_FULL.rsplit("_", 1)[0]
LLM_MATCHED = next(arm for arm in registered_arms if arm == f"{llm_prefix}_matched")
LLM_ARMS = (LLM_MATCHED, LLM_FULL)
for stream in STREAMS:
    assert set(selected[stream]["arm"]) == registered_arms
    scope = selected[stream][selected[stream]["arm"].isin(LLM_ARMS)]
    assert scope["model_name"].nunique() == 9
    assert scope.groupby("arm")["model_name"].nunique().eq(9).all()
    assert set(scope["model_name"]) == set(MODEL_NAMES)

## Frozen LLM scorer contract

Method: The block records the frozen LLM digest, deterministic settings, batch size 10, system prompt and output schema used for every score.

In [3]:
# non-economic-table
prompt_contract = {"system": SYSTEM, "user_line_template": "n{index}: {title}", "batch_protocol": identity["batch_protocol"]}
schema_contract = {"item": _ITEM, "envelope": "object keyed n0..n{batch_size-1}"}
assert _canonical_hash(prompt_contract) == identity["prompt_hash"]
assert _canonical_hash(schema_contract) == identity["schema_hash"]
technical_html = f'''<details><summary><b>Exact technical contract</b></summary>
<p><b>Frozen scorer digest:</b> {escape(identity["digest"][:16])}...; temperature 0; think=low; batch size 10.</p>
<p><b>System prompt</b></p><pre>{escape(SYSTEM)}</pre>
<p><b>User line:</b> <code>n{{index}}: {{title}}</code></p>
<p><b>Output schema</b></p><pre>{escape(json.dumps(_ITEM, indent=2))}</pre></details>'''
display(HTML(technical_html))
print("Takeaway: Indexed batches of ten preserve the link between each headline and its frozen score.")

Takeaway: Indexed batches of ten preserve the link between each headline and its frozen score.


## Matched and full feature sets

Method: LLM matched uses the same five causal sentiment fields for a scorer-only comparison; LLM full adds exactly three structured fields.

In [4]:
# non-economic-table
features = pd.DataFrame([
    ("LLM matched", "Same five causal fields: news sentiment, 24h news count, direct-event sentiment, news tone and macro releases; only the scorer changes"),
    ("LLM full", "LLM matched plus relevance decay, high-impact sentiment decay and 24h topic share"),
], columns=["Arm", "Feature difference"])
display(features)
print("Takeaway: The full arm changes only three declared structured fields, so the matched arm remains the scorer-only comparison.")

,Arm,Feature difference
0,LLM matched,"Same five causal fields: news sentiment, 24h n..."
1,LLM full,"LLM matched plus relevance decay, high-impact ..."


Takeaway: The full arm changes only three declared structured fields, so the matched arm remains the scorer-only comparison.


## H1 results for nine individual models

Method: Base, LLM matched and LLM full use the same six-month H1 calibration and are shown for all nine model families.

In [5]:
# economic-table
rows = []
labels = {"selected_base": "Price/VIX base", LLM_MATCHED: "LLM matched", LLM_FULL: "LLM full"}
for stream in STREAMS:
    scope = selected[stream][selected[stream]["arm"].isin(labels)].copy()
    scope["Index"] = stream.upper(); rows.append(scope)
h1 = pd.concat(rows, ignore_index=True).replace({"arm": labels}).rename(columns={
    "arm": "Features", "model_name": "Model", "width_bps": "Width bps", "tau": "Threshold",
    "eligible": "H1 status", "trades": "Trades", "n_long": "LONG", "n_short": "SHORT",
    "net_return": "Net %", "daily_sharpe": "Sharpe", "daily_sortino": "Sortino",
})
h1["Net %"] *= 100
h1["Model"] = h1["Model"].map(MODEL_LABELS)
h1["H1 status"] = h1["H1 status"].map({True: "Pass", False: "Below gate"})
h1 = h1.sort_values(["Net %", "Sortino", "Sharpe"], ascending=False)
display(h1[["Index", "Features", "Model", "Width bps", "Threshold", "H1 status", "Trades", "LONG", "SHORT", "Net %", "Sharpe", "Sortino"]].round(3))
best_h1 = h1.iloc[0]
print(f"Takeaway: {best_h1['Index']} {best_h1['Features']} {best_h1['Model']} has the highest H1 Net at {best_h1['Net %']:.2f}%.")

,Index,Features,Model,Width bps,Threshold,H1 status,Trades,LONG,SHORT,Net %,Sharpe,Sortino
27,USATECH,Price/VIX base,LogReg,10,0.65,Below gate,134,115,19,9.643,1.131,1.254
43,USATECH,LLM matched,LSTM,15,0.65,Pass,207,187,20,8.271,1.854,1.297
15,USA500,LLM matched,MLP,10,0.70,Below gate,332,314,18,7.546,1.274,1.241
12,USA500,LLM matched,Linear SVM,15,0.40,Below gate,401,360,41,6.644,0.601,0.450
24,USA500,LLM full,MLP,10,0.75,Below gate,170,148,22,6.489,1.694,1.871
44,USATECH,LLM matched,GRU,15,0.60,Below gate,379,337,42,5.240,0.704,0.553
0,USA500,Price/VIX base,LogReg,15,0.80,Below gate,205,119,86,3.353,0.978,0.427
32,USATECH,Price/VIX base,CatBoost,15,0.75,Pass,110,85,25,2.986,0.738,0.325
40,USATECH,LLM matched,XGBoost,10,0.75,Below gate,194,170,24,2.763,0.629,0.505
6,USA500,Price/VIX base,MLP,15,0.60,Below gate,725,571,154,2.722,0.373,0.248


Takeaway: USATECH Price/VIX base LogReg has the highest H1 Net at 9.64%.


## Forward results

Method: Every LLM matched and LLM full model retains its H1-selected width and threshold for the same July 2025-March 2026 descriptive economic comparison.

In [6]:
# forward-all-models
# economic-table
active = pd.concat([
    forward[stream][forward[stream]["arm"].isin(LLM_ARMS)].assign(Index=stream.upper())
    for stream in STREAMS
], ignore_index=True).replace({"arm": labels}).rename(columns={
    "arm": "Features", "model_name": "Model", "trades": "Trades", "trades_per_day": "Trades/day",
    "n_long": "LONG", "n_short": "SHORT", "net_return": "Net %", "daily_sharpe": "Sharpe",
    "daily_sortino": "Sortino", "max_drawdown": "Max drawdown %",
})
active["Model"] = active["Model"].map(MODEL_LABELS)
active["H1 status"] = active["h1_eligible"].map({True: "Pass", False: "Below gate"})
active["Forward status"] = active["status"].map({"traded": "Traded", "no_trades": "No trades"})
active["Net %"] *= 100; active["Max drawdown %"] *= 100
result_columns = ["Trades", "Trades/day", "LONG", "SHORT", "Net %", "Sharpe", "Sortino", "Max drawdown %"]
assert len(active) == 36 and active.groupby(["Index", "Features"])["Model"].nunique().eq(9).all()
assert np.isfinite(active[result_columns].to_numpy(dtype=float)).all()
active = active.sort_values(["Net %", "Sortino", "Model"], ascending=[False, False, True])
display(active[["Index", "Features", "Model", "H1 status", "Forward status", *result_columns]].round(3))
best = active.iloc[0]
print(f"Takeaway: {best['Index']} {best['Features']} {best['Model']} has the highest LLM Net at {best['Net %']:.2f}%; this secondary forward result is descriptive.")

,Index,Features,Model,H1 status,Forward status,Trades,Trades/day,LONG,SHORT,Net %,Sharpe,Sortino,Max drawdown %
1,USA500,LLM matched,Decision Tree,Below gate,Traded,152,0.555,152,0,3.499,1.465,0.829,-1.596
20,USATECH,LLM matched,Random Forest,Below gate,Traded,109,0.398,100,9,2.010,0.752,0.239,-1.784
29,USATECH,LLM full,Random Forest,Below gate,Traded,95,0.347,91,4,1.535,0.879,0.390,-1.181
9,USA500,LLM full,LogReg,Below gate,Traded,23,0.084,14,9,0.939,0.523,0.055,-1.671
6,USA500,LLM matched,MLP,Below gate,Traded,1,0.004,0,1,0.922,1.154,0.000,0.000
31,USATECH,LLM full,XGBoost,Pass,Traded,24,0.088,8,16,0.752,1.238,0.378,-0.349
34,USATECH,LLM full,LSTM,Pass,Traded,37,0.135,2,35,0.315,0.305,0.050,-0.675
24,USATECH,LLM matched,MLP,Below gate,Traded,2,0.007,0,2,0.280,0.380,0.044,-0.444
25,USATECH,LLM matched,LSTM,Pass,Traded,1,0.004,1,0,0.186,1.154,0.000,0.000
12,USA500,LLM full,Linear SVM,Below gate,Traded,21,0.077,11,10,0.119,0.058,0.008,-2.371


Takeaway: USA500 LLM matched Decision Tree has the highest LLM Net at 3.50%; this secondary forward result is descriptive.
